<a href="https://colab.research.google.com/github/kds825/education/blob/main/MNIST_%ED%95%99%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Sat Sep 26 17:50:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())

if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))

True
Tesla T4


In [5]:
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("사용 장치:", device)

x= torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])
print("처음 위치:", x.device)
x=x.to(device)
print("이동 후 위치:", x.device)


사용 장치: cuda
처음 위치: cpu
이동 후 위치: cuda:0


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [11]:
from torchvision import datasets, transforms
transform = transforms.ToTensor()
print(transform)

ToTensor()


In [15]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform

)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("학습 데이터 개수:", len(train_dataset))
print("테스트 데이터 개수:", len(test_dataset))

학습 데이터 개수: 60000
테스트 데이터 개수: 10000


In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [16]:
class SimpleModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.network = nn.Sequential(
        nn.Linear(28*28, 128),
        nn.ReLU(),
        nn.Linear(128, 10)

    )
  def forward(self, x):
    x= self.flatten(x)
    x= self.network(x)
    return x



In [18]:
model = SimpleModel()
print(model)

SimpleModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [19]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model=model.to(device)
print("사용 장치:",device)

사용 장치: cuda


In [20]:
criterion = nn.CrossEntropyLoss()


In [21]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [26]:
epochs=10
for epoch in range(epochs):
  total_loss=0
  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss=criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(
      f"Epoch {epoch+1}/{epochs},"
      f"Loss: {total_loss / len(train_loader):.4f}"
  )


Epoch 1/10,Loss: 0.1107
Epoch 2/10,Loss: 0.0887
Epoch 3/10,Loss: 0.0740
Epoch 4/10,Loss: 0.0615
Epoch 5/10,Loss: 0.0532
Epoch 6/10,Loss: 0.0458
Epoch 7/10,Loss: 0.0383
Epoch 8/10,Loss: 0.0341
Epoch 9/10,Loss: 0.0281
Epoch 10/10,Loss: 0.0244


In [27]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (
        predicted == labels
    ).sum().item()
accuracy = 100*correct / total
print("정확도:", accuracy, "%")

정확도: 97.77 %
